# Análise exploratória — chamados de Help Desk

Notebook de prática com a base fictícia tratada. A análise descreve a amostra; os resultados não representam uma empresa real.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
dados = pd.read_csv(
    Path('data/chamados_help_desk_tratados.csv'),
    encoding='utf-8-sig',
    parse_dates=['data_abertura', 'data_encerramento']
)
fechados = dados.loc[dados['status'].eq('Fechado')].copy()
dados.shape, dados.head()

## Volume mensal

Agrupamos pela data de abertura para observar a variação do volume entre meses. Um único ano não permite afirmar que existe sazonalidade.

In [ ]:
mensal = (
    dados.assign(mes=dados['data_abertura'].dt.to_period('M').astype(str))
    .groupby('mes').size().rename('chamados').reset_index()
)
sns.lineplot(data=mensal, x='mes', y='chamados', marker='o')
plt.xticks(rotation=45)
plt.title('Chamados abertos por mês')
plt.tight_layout()

## Categorias e departamentos

As barras horizontais facilitam comparar os grupos mais e menos frequentes.

In [ ]:
ordem = dados['categoria'].value_counts().sort_values().index
sns.countplot(data=dados, y='categoria', order=ordem)
plt.title('Chamados por categoria')
plt.tight_layout()

## Tempo de resolução

O boxplot usa apenas chamados fechados. Os pontos afastados são valores que merecem investigação; não devem ser removidos sem entender o contexto.

In [ ]:
sns.boxplot(data=fechados, x='prioridade', y='tempo_resolucao_horas',
            order=['Crítica', 'Alta', 'Média', 'Baixa'])
plt.title('Tempo de resolução por prioridade')
plt.tight_layout()
fechados['tempo_resolucao_horas'].describe()

In [ ]:
satisfacao = fechados['satisfacao_usuario'].value_counts().sort_index()
sns.barplot(x=satisfacao.index.astype(str), y=satisfacao.values)
plt.title('Distribuição da satisfação')
plt.xlabel('Nota de 1 a 5')
plt.ylabel('Quantidade de respostas')
plt.tight_layout()